# D2C Skincare Subscription Analytics â€” Data Cleaning & Audit

## Objective

This notebook will:
- Audit the intentionally messy raw data
- Clean and standardize the data
- Validate data quality
- Export cleaned data for analysis

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define data paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / 'data/raw'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data/processed'

# Create processed directory if it doesn't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f'Raw data directory: {RAW_DATA_DIR.resolve()}')
print(f'Processed data directory: {PROCESSED_DATA_DIR.resolve()}')

Raw data directory: D:\cooding_playground\data analysist projects\D2C-Skincare-Subscription-Analytics\data\raw
Processed data directory: D:\cooding_playground\data analysist projects\D2C-Skincare-Subscription-Analytics\data\processed


In [3]:
# Load raw CSV files
users = pd.read_csv(RAW_DATA_DIR / 'users.csv')
subscription_events = pd.read_csv(RAW_DATA_DIR / 'subscription_events.csv')
orders = pd.read_csv(RAW_DATA_DIR / 'orders.csv')
marketing_spend = pd.read_csv(RAW_DATA_DIR / 'marketing_spend.csv')

print('All raw datasets loaded successfully.')

All raw datasets loaded successfully.


In [4]:
# Display basic information for each dataset
print('=== Raw Data Summary ===')
print(f'\nUsers:')
print(f'  Rows: {len(users):,}')
print(f'  Columns: {list(users.columns)}')

print(f'\nSubscription Events:')
print(f'  Rows: {len(subscription_events):,}')
print(f'  Columns: {list(subscription_events.columns)}')

print(f'\nOrders:')
print(f'  Rows: {len(orders):,}')
print(f'  Columns: {list(orders.columns)}')

print(f'\nMarketing Spend:')
print(f'  Rows: {len(marketing_spend):,}')
print(f'  Columns: {list(marketing_spend.columns)}')

=== Raw Data Summary ===

Users:
  Rows: 60,600
  Columns: ['user_id', 'signup_date', 'acquisition_channel', 'city', 'email']

Subscription Events:
  Rows: 117,058
  Columns: ['user_id', 'event_date', 'event_type']

Orders:
  Rows: 86,364
  Columns: ['order_id', 'user_id', 'order_date', 'order_value', 'plan_type', 'status']

Marketing Spend:
  Rows: 138
  Columns: ['month', 'acquisition_channel', 'spend_inr', 'new_users_acquired']


## Raw Data Audit

Read-only audit of the four raw DataFrames (`users`, `subscription_events`, `orders`, `marketing_spend`).
**No cleaning happens in this section** - no duplicates are dropped, no nulls are filled, no channels or dates are standardized.

Checks performed for every table:
1. Row count
2. Column names
3. Data types
4. Null count and null % by column (plus total nulls)
5. Duplicate row count

Table-specific checks:
6. `users` - duplicate `user_id` count and acquisition-channel distribution
7. `subscription_events` - mixed date-format evidence and null `event_type` count
8. `orders` - `user_id` values that do not exist in `users.user_id` and `order_value <= 0`
9. `marketing_spend` - missing month/channel combinations vs the expected 36 months x 4 canonical channels

The section ends with a concise before-cleaning audit summary table.

In [5]:
# ------------------------------------------------------------------
# Items 1-5: generic per-table audit (READ-ONLY).
# Nothing is modified: no duplicates dropped, no nulls filled.
# ------------------------------------------------------------------

def audit_dataframe(df, name):
    """Print a structural audit of a raw DataFrame and return summary stats."""
    row_count = len(df)
    col_names = list(df.columns)
    null_counts = df.isnull().sum()
    total_nulls = int(null_counts.sum())
    total_cells = row_count * len(col_names)
    duplicate_rows = int(df.duplicated().sum())

    print(f'=== {name} ===')
    print(f'1) Row count: {row_count:,}')
    print(f'2) Column names: {col_names}')
    print('3) Data types:')
    print(df.dtypes.to_string())

    null_pct = (null_counts / row_count * 100).round(2) if row_count > 0 else null_counts.astype(float)
    print('4) Null count / null % by column:')
    print(pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct}).to_string())
    overall_pct = (total_nulls / total_cells * 100) if total_cells else 0.0
    print(f'   Total nulls: {total_nulls:,} ({overall_pct:.2f}% of all cells)')

    print(f'5) Duplicate rows: {duplicate_rows:,}')

    return {
        'rows': row_count,
        'n_columns': len(col_names),
        'total_nulls': total_nulls,
        'null_pct_of_cells': round(overall_pct, 2),
        'duplicate_rows': duplicate_rows,
    }

RAW_TABLES = {
    'users': users,
    'subscription_events': subscription_events,
    'orders': orders,
    'marketing_spend': marketing_spend,
}

table_audits = {name: audit_dataframe(df, name) for name, df in RAW_TABLES.items()}


=== users ===
1) Row count: 60,600
2) Column names: ['user_id', 'signup_date', 'acquisition_channel', 'city', 'email']
3) Data types:
user_id                str
signup_date            str
acquisition_channel    str
city                   str
email                  str
4) Null count / null % by column:
                     null_count  null_pct
user_id                       0       0.0
signup_date                   0       0.0
acquisition_channel           0       0.0
city                       3028       5.0
email                         0       0.0
   Total nulls: 3,028 (1.00% of all cells)
5) Duplicate rows: 600
=== subscription_events ===
1) Row count: 117,058
2) Column names: ['user_id', 'event_date', 'event_type']
3) Data types:
user_id       str
event_date    str
event_type    str
4) Null count / null % by column:
            null_count  null_pct
user_id              0      0.00
event_date           0      0.00
event_type        2354      2.01
   Total nulls: 2,354 (0.67% of all c

In [6]:
# ------------------------------------------------------------------
# Table-specific integrity checks (READ-ONLY).
# ------------------------------------------------------------------

# 6) users - duplicate user_id count and acquisition-channel distribution
users_dup_id_rows = int(users['user_id'].duplicated().sum())  # extra copies beyond the first occurrence
users_dup_id_distinct = int(users.loc[users['user_id'].duplicated(), 'user_id'].nunique())
print('6) users - duplicate user_id and acquisition-channel checks')
print(f'   Rows with an already-seen user_id: {users_dup_id_rows:,}')
print(f'   Distinct user_ids affected:         {users_dup_id_distinct:,}')
print('   Acquisition-channel distribution:')
print(users['acquisition_channel'].value_counts(dropna=False).to_string())

# 7) subscription_events - mixed date-format evidence and null event_type count
subscription_event_dates = subscription_events['event_date'].astype(str)
subscription_date_format_counts = pd.Series(
    np.select(
        [
            subscription_event_dates.str.match(r'^\d{4}-\d{2}-\d{2}'),
            subscription_event_dates.str.match(r'^\d{2}/\d{2}/\d{4}'),
        ],
        ['ISO-like YYYY-MM-DD', 'slash-like DD/MM/YYYY'],
        default='other or unparseable',
    )
).value_counts()
subscription_null_event_types = int(subscription_events['event_type'].isna().sum())
print('\n7) subscription_events - date-format and null event_type checks')
print('   Date-format evidence:')
print(subscription_date_format_counts.to_string())
print(f'   Null event_type rows: {subscription_null_event_types:,}')

# 8) orders - orphan user_ids and non-positive order values
orphan_mask = ~orders['user_id'].isin(users['user_id'])
orphan_rows = int(orphan_mask.sum())
orphan_id_count = int(orders.loc[orphan_mask, 'user_id'].nunique())
print('\n8) orders - orphan user_id check (referential integrity)')
print(f'   Order rows whose user_id is NOT in users: {orphan_rows:,}')
print(f'   Distinct unknown user_ids referenced:     {orphan_id_count:,}')

nonpositive_mask = orders['order_value'] <= 0
nonpositive_rows = int(nonpositive_mask.sum())
print('\n   orders - order_value <= 0 check')
print(f'   Rows with order_value <= 0: {nonpositive_rows:,}')
if nonpositive_rows > 0:
    print(orders.loc[nonpositive_mask, ['order_id', 'user_id', 'order_value']].head(5).to_string(index=False))

# 9) marketing_spend - missing month/channel combinations vs the expected grid.
EXPECTED_MONTHS = pd.period_range('2025-01', '2027-12', freq='M')
EXPECTED_CHANNELS = [
    'Referral', 'Instagram Ads', 'Google Ads', 'Organic',
]
expected_pairs = {(m, c) for m in EXPECTED_MONTHS for c in EXPECTED_CHANNELS}

parsed_months = pd.to_datetime(marketing_spend['month'], errors='coerce')
unparseable_months = int(parsed_months.isna().sum())
actual_pairs = set(zip(parsed_months.dt.to_period('M'), marketing_spend['acquisition_channel']))
missing_pairs = sorted(expected_pairs - actual_pairs, key=lambda p: (str(p[0]), str(p[1])))
unexpected_pairs = sorted(actual_pairs - expected_pairs, key=lambda p: (str(p[0]), str(p[1])))
dup_combo_rows = int(marketing_spend.duplicated(subset=['month', 'acquisition_channel']).sum())

print('\n9) marketing_spend - month/channel completeness check')
print(f'   Expected grid: {len(EXPECTED_MONTHS)} months x {len(EXPECTED_CHANNELS)} channels = {len(expected_pairs):,} combinations')
print(f'   Actual rows: {len(marketing_spend):,} covering {len(actual_pairs):,} distinct combinations')
print(f'   Missing month/channel combinations: {len(missing_pairs):,}')
if missing_pairs:
    print(pd.DataFrame(missing_pairs, columns=['month', 'acquisition_channel']).to_string(index=False))
print(f'   Unexpected (out-of-grid) combinations: {len(unexpected_pairs):,}')
print(f'   Duplicate month/channel rows within the table: {dup_combo_rows:,}')
print(f'   Unparseable month values: {unparseable_months:,}')

6) users - duplicate user_id and acquisition-channel checks
   Rows with an already-seen user_id: 600
   Distinct user_ids affected:         600
   Acquisition-channel distribution:
acquisition_channel
Referral         15246
Instagram Ads    15192
Google Ads       15130
Organic          15032

7) subscription_events - date-format and null event_type checks
   Date-format evidence:
ISO-like YYYY-MM-DD      114745
slash-like DD/MM/YYYY      2313
   Null event_type rows: 2,354

8) orders - orphan user_id check (referential integrity)
   Order rows whose user_id is NOT in users: 431
   Distinct unknown user_ids referenced:     431

   orders - order_value <= 0 check
   Rows with order_value <= 0: 1,295
  order_id user_id  order_value
ORD0000227 U000148          0.0
ORD0000266 U000169          0.0
ORD0000343 U000222          0.0
ORD0000348 U000225          0.0
ORD0000380 U000242       -100.0

9) marketing_spend - month/channel completeness check
   Expected grid: 36 months x 4 channels = 14

In [7]:
# ------------------------------------------------------------------
# Concise audit summary - main before-cleaning counts (READ-ONLY).
# 'NaN' means the check is not applicable to that table.
# ------------------------------------------------------------------

table_order = ['users', 'subscription_events', 'orders', 'marketing_spend']
audit_summary = pd.DataFrame(
    {
        'rows':            [table_audits[t]['rows'] for t in table_order],
        'columns':         [table_audits[t]['n_columns'] for t in table_order],
        'total_nulls':     [table_audits[t]['total_nulls'] for t in table_order],
        'null_pct_cells':  [table_audits[t]['null_pct_of_cells'] for t in table_order],
        'duplicate_rows':  [table_audits[t]['duplicate_rows'] for t in table_order],
        'dup_user_id_rows':             [users_dup_id_rows, np.nan, np.nan, np.nan],
        'orphan_user_id_rows':          [np.nan, np.nan, orphan_rows, np.nan],
        'order_value_le_0_rows':        [np.nan, np.nan, nonpositive_rows, np.nan],
        'missing_month_channel_combos': [np.nan, np.nan, np.nan, len(missing_pairs)],
    },
    index=table_order,
)
print('=== Raw Data Audit Summary (before cleaning) ===')
audit_summary


=== Raw Data Audit Summary (before cleaning) ===


,rows,columns,total_nulls,null_pct_cells,duplicate_rows,dup_user_id_rows,orphan_user_id_rows,order_value_le_0_rows,missing_month_channel_combos
users,60600,5,3028,1.00,600,600.0,NaN,NaN,NaN
subscription_events,117058,3,2354,0.67,0,NaN,NaN,NaN,NaN
orders,86364,6,0,0.00,0,NaN,431.0,1295.0,NaN
marketing_spend,138,4,0,0.00,0,NaN,NaN,NaN,6.0


## Level 2C + 2D: Channel and Date Normalization

This section creates cleaned copies of the raw tables. Level 2C standardizes only known acquisition-channel variants to the four canonical labels and leaves unknown non-null values unchanged for reporting. Level 2D parses the targeted date columns with pandas, converting invalid or missing values to `NaT`. No deduplication, orphan handling, invalid-order-value handling, or marketing completeness repair is performed here.

In [8]:
# ------------------------------------------------------------------
# Level 2C: standardize acquisition_channel on cleaned copies.
# ------------------------------------------------------------------

CANONICAL_CHANNELS = [
    'Referral',
    'Instagram Ads',
    'Google Ads',
    'Organic',
]

CHANNEL_VARIANTS = {
    'referral': 'Referral',
    'instagram ads': 'Instagram Ads',
    'instagram': 'Instagram Ads',
    'google ads': 'Google Ads',
    'google': 'Google Ads',
    'organic': 'Organic',
}


def standardize_channel(value):
    if pd.isna(value):
        return value
    normalized = ' '.join(str(value).strip().split()).lower()
    return CHANNEL_VARIANTS.get(normalized, value)


users_clean = users.copy()
marketing_spend_clean = marketing_spend.copy()

for frame in (users_clean, marketing_spend_clean):
    frame['acquisition_channel'] = frame['acquisition_channel'].map(standardize_channel)

unknown_user_channels = sorted(
    set(users_clean['acquisition_channel'].dropna()) - set(CANONICAL_CHANNELS)
)
unknown_marketing_channels = sorted(
    set(marketing_spend_clean['acquisition_channel'].dropna()) - set(CANONICAL_CHANNELS)
)

print('=== Level 2C: Acquisition Channel Standardization ===')
print('Known variants are matched case-insensitively after trimming and collapsing spaces.')
print('Unknown non-null values are preserved and reported, not coerced to a canonical channel.')
print(f'Users unknown channels: {unknown_user_channels}')
print(f'Marketing unknown channels: {unknown_marketing_channels}')

=== Level 2C: Acquisition Channel Standardization ===
Known variants are matched case-insensitively after trimming and collapsing spaces.
Unknown non-null values are preserved and reported, not coerced to a canonical channel.
Users unknown channels: []
Marketing unknown channels: []


In [ ]:
# ------------------------------------------------------------------
# Level 2D: normalize targeted date columns on cleaned copies.
# ------------------------------------------------------------------

DATE_COLUMNS = {
    'users.signup_date': (users_clean, 'signup_date'),
    'subscription_events.event_date': (subscription_events.copy(), 'event_date'),
    'orders.order_date': (orders.copy(), 'order_date'),
    'marketing_spend.month': (marketing_spend_clean, 'month'),
}

cleaned_tables = {
    'users': users_clean,
    'subscription_events': DATE_COLUMNS['subscription_events.event_date'][0],
    'orders': DATE_COLUMNS['orders.order_date'][0],
    'marketing_spend': marketing_spend_clean,
}

SIMULATION_START = pd.Timestamp('2025-01-01')
SIMULATION_END = pd.Timestamp('2027-12-31')
parse_failures = {}
out_of_range_dates = {}

for table_name, (frame, column_name) in DATE_COLUMNS.items():
    parsed_dates = pd.to_datetime(
        frame[column_name],
        errors='coerce',
        format='mixed',
    )
    parse_failures[table_name] = int(parsed_dates.isna().sum())
    out_of_range_mask = parsed_dates.notna() & ~parsed_dates.between(SIMULATION_START, SIMULATION_END)
    out_of_range_dates[table_name] = int(out_of_range_mask.sum())
    parsed_dates.loc[out_of_range_mask] = pd.NaT
    frame[column_name] = parsed_dates

# Keep monthly marketing values as month-start timestamps for grouping.
cleaned_tables['marketing_spend']['month'] = cleaned_tables['marketing_spend']['month'].dt.to_period('M').dt.to_timestamp()

print('=== Level 2D: Date Normalization ===')
print('Date parsing uses pandas with errors=coerce; invalid or missing values become NaT.')
print('Dates outside the locked simulation window are treated as invalid domain dates and become NaT.')
print(f'Unparseable or missing before range validation: {parse_failures}')
print(f'Out-of-range dates converted to NaT: {out_of_range_dates}')

=== Level 2D: Date Normalization ===
Date parsing uses pandas with errors=coerce; invalid or missing values become NaT.
Parsing failures / NaT counts: {'users.signup_date': 0, 'subscription_events.event_date': 0, 'orders.order_date': 0, 'marketing_spend.month': 0}


In [ ]:
# ------------------------------------------------------------------
# Compact validation for Level 2C and Level 2D.
# ------------------------------------------------------------------

for table_name, frame in {
    'users': users_clean,
    'marketing_spend': marketing_spend_clean,
}.items():
    non_null_channels = frame['acquisition_channel'].dropna()
    unexpected_channels = sorted(set(non_null_channels) - set(CANONICAL_CHANNELS))
    print(f'=== {table_name} channel validation ===')
    print(f'Unique channels: {sorted(non_null_channels.unique())}')
    print('Counts by channel:')
    print(frame['acquisition_channel'].value_counts(dropna=False).to_string())
    print(f'All non-null channels canonical: {not unexpected_channels}')
    if unexpected_channels:
        print(f'Unexpected channels: {unexpected_channels}')

print('\n=== Date validation ===')
for table_name, (frame, column_name) in DATE_COLUMNS.items():
    valid_dates = frame[column_name].dropna()
    print(f'{table_name}: dtype={frame[column_name].dtype}, min={valid_dates.min()}, max={valid_dates.max()}, NaT={frame[column_name].isna().sum():,}')

subscription_event_dates = cleaned_tables['subscription_events']['event_date']
all_date_types_consistent = all(
    pd.api.types.is_datetime64_any_dtype(frame[column_name])
    for frame, column_name in DATE_COLUMNS.values()
)
print(
    'subscription_events.event_date consistently parsed: '
    f'{pd.api.types.is_datetime64_any_dtype(subscription_event_dates)}'
)
print(f'All targeted date columns consistently parsed: {all_date_types_consistent}')
print(f'Raw row counts preserved: {all(len(cleaned_tables[name]) == len(RAW_TABLES[name]) for name in RAW_TABLES)}')

=== users channel validation ===
Unique channels: ['Google Ads', 'Instagram Ads', 'Organic', 'Referral']
Counts by channel:
acquisition_channel
Referral         15246
Instagram Ads    15192
Google Ads       15130
Organic          15032
All non-null channels canonical: True
=== marketing_spend channel validation ===
Unique channels: ['Google Ads', 'Instagram Ads', 'Organic', 'Referral']
Counts by channel:
acquisition_channel
Google Ads       35
Organic          35
Referral         34
Instagram Ads    34
All non-null channels canonical: True

=== Date validation ===
users.signup_date: dtype=datetime64[us], min=2025-01-01 00:00:00, max=2027-12-31 00:00:00, NaT=0
subscription_events.event_date: dtype=datetime64[us], min=2025-01-01 00:00:00, max=2028-11-02 00:00:00, NaT=0
orders.order_date: dtype=datetime64[us], min=2025-01-09 00:00:00, max=2027-12-31 00:00:00, NaT=0
marketing_spend.month: dtype=datetime64[us], min=2025-01-01 00:00:00, max=2027-12-01 00:00:00, NaT=0
subscription_events.even